# M05-01 — Ranking

Referencia de validación. El alumno trabaja en `notebooks/alumno/M05-01-ranking-ventana.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
from pyspark.sql.functions import col, row_number, sum as fsum
from pyspark.sql.window import Window
spark = get_spark("novashop-m05")
cust = spark.read.parquet(str(STAGING / "customer_gmv"))
w_global = Window.orderBy(col("gmv").desc())
top10 = cust.withColumn("rn", row_number().over(w_global)).where(col("rn") <= 10)
top10.orderBy("rn").show()
assert top10.count() == 10
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
product_gmv = (
    fact.join(customers, "customer_id", "inner").where(col("is_billable"))
    .groupBy("customer_id", "product_id").agg(fsum("gmv_line").alias("gmv"))
)
w_prod = Window.partitionBy("customer_id").orderBy(col("gmv").desc())
top3 = product_gmv.withColumn("rn", row_number().over(w_prod)).where(col("rn") <= 3)
print("filas top3", top3.count())
assert top3.count() <= 211 * 3
print("M05-01 OK")
